# aoi-sentinel — Colab Quickstart

Mamba-RL false-call filter for SMT AOI on a free Colab T4 (or Pro L4/A100).

## What you'll run
1. GPU sanity check + clone repo
2. Install deps (torch + mamba-ssm + project)
3. Download VisA PCB benchmark data
4. **Stage 0** — supervised pretraining of MambaVision on VisA (~30–60 min on T4)
5. **Stage 1** — Lagrangian PPO on the NPI simulator (~30–60 min)
6. Plot cost / escape / λ trajectories

**Runtime → Change runtime type → GPU** before running.

Tip: free Colab disconnects after ~12h idle. Mount Google Drive and point checkpoint paths there if you want persistence.

## 1. GPU + repo

In [ ]:
!nvidia-smi | head -16

In [ ]:
import os
if not os.path.exists('/content/aoi-sentinel'):
    !git clone https://github.com/DrJinHoChoi/aoi-sentinel.git /content/aoi-sentinel
%cd /content/aoi-sentinel
!git pull

## 2. Install

Colab ships with a recent torch + CUDA 12.x. We install Mamba kernels with `--no-build-isolation` so they pick up the existing torch.

First install often takes 5–10 minutes (compiling CUDA). It's cached for subsequent runs.

In [ ]:
import torch, sys
print('python:', sys.version.split()[0])
print('torch :', torch.__version__, '| cuda:', torch.version.cuda, '| device:', torch.cuda.get_device_name(0))

In [ ]:
# Mamba CUDA kernels. --no-build-isolation reuses Colab's torch.
!pip install -q packaging ninja
!pip install -q causal-conv1d==1.4.0 --no-build-isolation
!pip install -q mamba-ssm==2.2.2 --no-build-isolation

In [ ]:
# Project + the rest. We skip torch in the train extras since Colab already has one.
!pip install -q -e ".[train,dev]" \
    --no-deps  # install our metadata; we control deps below
!pip install -q timm==1.0.11 mambavision albumentations gymnasium tensorboard wandb \
    rich click pyyaml lxml pyarrow pandas opencv-python-headless

In [ ]:
# Sanity check: both Mamba surfaces import and run on GPU.
import torch
from mamba_ssm import Mamba
import timm

vm = timm.create_model('mambavision_tiny_1k', pretrained=True).cuda().eval()
x = torch.randn(2, 3, 224, 224).cuda()
with torch.no_grad():
    print('vision out:', vm(x).shape)

mb = Mamba(d_model=128).cuda().eval()
y = torch.randn(2, 64, 128).cuda()
with torch.no_grad():
    print('seq out   :', mb(y).shape)

## 3. Download benchmark data

VisA full archive is ~4.5 GB. We grab it once and prune to the 4 PCB classes (~1.2 GB).

**Optional**: mount Google Drive and copy `data/raw/visa/` there to skip re-downloading on future sessions.

In [ ]:
# Optional Drive mount — uncomment if you want checkpoint/data persistence.
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
!python scripts/download_visa.py --out data/raw/visa --pcb-only

In [ ]:
!ls data/raw/visa/ && echo '---' && ls data/raw/visa/pcb1/Data/Images/ 2>/dev/null | head

## 4. Stage 0 — pretrain MambaVision on VisA

Cost-sensitive supervised classifier. Output: image-encoder weights for the RL stage.

In [ ]:
!aoi train pretrain --config configs/stage0_pretrain_colab.yaml

## 5. Stage 1 — Mamba RL on the NPI simulator

Lagrangian PPO under the escape-rate budget. Watch for:
- `avg_cost` decreasing over iterations
- `λ` rising when escape budget is violated, falling when satisfied
- `cum_escape` should stay near 0

In [ ]:
!aoi train npi-rl --config configs/stage1_npi_rl_colab.yaml 2>&1 | tee /content/stage1.log

## 6. Plot trajectories

In [ ]:
import re
import matplotlib.pyplot as plt

rows = []
pat = re.compile(r'iter (\d+).*?λ=([\-\d\.]+).*?avg_cost=([\-\d\.]+).*?cum_cost=([\-\d\.]+).*?cum_escape=(\d+)')
with open('/content/stage1.log') as f:
    for line in f:
        m = pat.search(line)
        if m:
            rows.append([int(m[1]), float(m[2]), float(m[3]), float(m[4]), int(m[5])])

import numpy as np
rows = np.array(rows)
fig, axes = plt.subplots(1, 4, figsize=(18, 3.5))
for ax, col, name in zip(axes, [1, 2, 3, 4], ['λ (Lagrange)', 'avg cost / step', 'cumulative cost', 'cumulative escapes']):
    ax.plot(rows[:, 0], rows[:, col])
    ax.set_xlabel('iter'); ax.set_title(name); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()

## Next

- Run with `--config configs/stage1_npi_rl.yaml` (full size) on a Colab Pro A100 / L4 for the proper baseline numbers.
- Add DeepPCB and SolDef_AI to the data mix (`configs/.../data.dataset`).
- When real Saki data arrives, swap `aoi_sentinel.data.benchmarks` for `aoi_sentinel.data.saki` and rerun stage 1 with `init_from` pointing at the stage 0 checkpoint.